# Coleta de Dados na Web: do Scraper ao RAG
## Notebook do Estudo de Caso — Corpus de Governança de IA

> **Material didático.** Este notebook acompanha a aula sobre coleta de dados na web. Ele constrói,
> passo a passo, um *scraper* que varre um conjunto de páginas selecionadas e produz um corpus textual
> limpo — o tipo de matéria-prima que alimenta um sistema **RAG** (Retrieval-Augmented Generation).

**Como usar este notebook em aula:**
- As seções seguem os blocos da aula. Cada uma começa com uma explicação e termina em código executável.
- Procure os blocos **`PARE E PENSE`**: são perguntas para discutir antes de rodar a célula seguinte.
- Os **`EXPERIMENTE`** indicam onde os exercícios da prática se encaixam.

**O cenário.** Uma equipe precisa criar um chat conversacional para auxilar pesquisadores, respondendo perguntas sobre *governança e regulação de IA em saúde* para elaboração de projetos que estejam de acordo com princípios de uso de IA responsável.
As fontes (NIST, União Europeia, ISO, OMS, OCDE, ANPD...) estão numa planilha curada. Vamos transformá-las
em um corpus estruturado.


---
## Bloco 1 — Fundamentos: requisição e *parsing*

Toda coleta estática tem dois passos: (1) **baixar** o HTML com uma requisição HTTP e (2) **analisar**
(*parse*) esse HTML para extrair o que interessa. Usamos `requests` para o primeiro e `BeautifulSoup`
para o segundo.

> **PARE E PENSE.** Por que `requests` + `BeautifulSoup` **não** conseguem ver conteúdo que só aparece
> depois que o JavaScript roda no navegador? O que isso diz sobre os limites do scraping estático?


In [ ]:
# No Colab estas bibliotecas já existem. Se necessário, descomente:
# !pip install requests beautifulsoup4 truststore

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
from collections import deque, Counter
import time
import json
import csv
import io
import re

try:
    import truststore
except ImportError:
    truststore = None
else:
    truststore.inject_into_ssl()
    print('Store de certificados do Windows habilitado para SSL.')

print('Ambiente pronto.')

Ambiente pronto.


In [15]:
import sys
import ssl

print("Python:", sys.executable)
print("OpenSSL:", ssl.OPENSSL_VERSION)
print("Verify paths:", ssl.get_default_verify_paths())

try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore: OK")
except Exception as e:
    print("truststore: FALHOU ->", e)

import requests

url = "https://www.iso.org/robots.txt"
try:
    r = requests.get(url, timeout=10)
    print("HTTPS test: OK")
    print("status:", r.status_code)
    print("content-type:", r.headers.get("Content-Type"))
except Exception as e:
    print("HTTPS test: FALHOU ->", type(e).__name__)
    print(e)

Python: c:\Users\leonardo.flores\AppData\Local\anaconda3\python.exe
OpenSSL: OpenSSL 3.0.18 30 Sep 2025
Verify paths: DefaultVerifyPaths(cafile='C:\\programdata\\mdm\\CACertificates\\ca_bundle.pem', capath=None, openssl_cafile_env='SSL_CERT_FILE', openssl_cafile='C:\\Program Files\\Common Files\\ssl/cert.pem', openssl_capath_env='SSL_CERT_DIR', openssl_capath='C:\\Program Files\\Common Files\\ssl/certs')
truststore: OK
HTTPS test: OK
status: 200
content-type: text/plain


### Anatomia de uma requisição

Vamos baixar uma página e inspecionar a resposta: o **código de status** (200 = OK, 403 = proibido,
404 = não encontrado) e o **tipo de conteúdo**. Sempre verificamos isso antes de confiar na resposta.

In [16]:
# Identificamos nosso robô honestamente via User-Agent
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; ScraperDidatico/1.0; +https://example.org/bot-info)'
}

url_exemplo = 'https://www.iso.org/obp/ui/en/#iso:std:iso-iec:5338:ed-1:v1:en'
# 'https://www.iana.org/help/example-domains'
# teste este também:

try:
    resp = requests.get(url_exemplo, headers=HEADERS, timeout=10)
    print('Status:', resp.status_code)
    print('Content-Type:', resp.headers.get('Content-Type'))
    print('Tamanho do HTML:', len(resp.text), 'caracteres')

    soup = BeautifulSoup(resp.text, 'html.parser')
    print('Título da página:', soup.title.string if soup.title else '(sem título)')
except requests.exceptions.SSLError as e:
    print('[SSL] Falha ao validar o certificado do servidor.')
    print('     Isso impede a conexão antes de existir resposta HTTP.')
    print('     Detalhe:', e)
except requests.RequestException as e:
    print('[ERRO] Falha de rede/HTTP ao acessar a página.')
    print('       Detalhe:', e)

Status: 200
Content-Type: text/html;charset=utf-8
Tamanho do HTML: 1791 caracteres
Título da página: (sem título)


---
## Bloco 2 — Ética e legalidade

Antes de coletar em escala, três compromissos: respeitar o **robots.txt**, **pausar** entre requisições
(*rate limiting*) e **identificar** nosso robô. O princípio que guia tudo:

> *Conseguir tecnicamente não é o mesmo que ter permissão.*

O `robots.txt` é um arquivo que cada site publica dizendo o que robôs podem acessar. A biblioteca padrão
do Python já sabe lê-lo.

In [17]:
_robots_cache = {}  # guarda o parser de cada domínio (não rebaixar o robots.txt toda hora)


def pode_acessar(url, respeitar_robots=True):
    """Consulta o robots.txt do domínio e diz se nosso User-Agent pode acessar a URL.

    Lemos o robots.txt com `requests` (mesmo User-Agent e um timeout explícito),
    em vez de deixar o RobotFileParser baixar sozinho. Sem timeout, um site lento
    poderia travar o crawl; e sem o nosso User-Agent, alguns servidores recusam a
    própria leitura do robots.txt. Em produção, esta é a forma recomendada.
    """
    if not respeitar_robots:
        return True
    p = urlparse(url)
    dominio = f'{p.scheme}://{p.netloc}'
    if dominio not in _robots_cache:
        rp = RobotFileParser()
        robots_url = urljoin(dominio, '/robots.txt')
        try:
            resp = requests.get(robots_url, headers=HEADERS, timeout=10)
            if resp.status_code >= 400:
                rp = None  # 404/403 etc.: sem robots.txt utilizável -> seguimos com cautela
            else:
                rp.parse(resp.text.splitlines())
        except requests.exceptions.SSLError as e:
            print(f'  [SSL] não foi possível validar o certificado ao ler {robots_url}')
            print(f'       {e}')
            return False
        except requests.RequestException:
            rp = None  # falha de rede ao ler o robots.txt -> seguimos com cautela
        _robots_cache[dominio] = rp
    rp = _robots_cache[dominio]
    return True if rp is None else rp.can_fetch(HEADERS['User-Agent'], url)

# Exemplo: muitos sites grandes restringem partes do site
for teste in ['https://www.iso.org/standard/81230.html',
              'https://unesdoc.unesco.org/ark:/48223/pf0000386276']:
    print(f'{"permitido" if pode_acessar(teste) else "BLOQUEADO":10} {teste}')

permitido  https://www.iso.org/standard/81230.html
permitido  https://unesdoc.unesco.org/ark:/48223/pf0000386276


> **PARE E PENSE.** O que o scraper **deve** fazer nos casos em que há retorno 403? (Dica: a resposta correta nunca é “forçar”.)


---
## Bloco 3 — Arquitetura do crawler

Agora as funções que sustentam o crawler. Primeiro, **normalizar URLs** (resolver links relativos,
remover âncoras) e **comparar domínios**. Sem isso, o mesmo recurso apareceria como várias URLs
diferentes e cairíamos em ciclos.

In [18]:
def normalizar_url(base, link):
    """Converte um link (relativo ou absoluto) em URL absoluta e limpa (sem âncora)."""
    if not link:
        return None
    url = urljoin(base, link.strip())
    p = urlparse(url)
    if p.scheme not in ('http', 'https'):   # ignora mailto:, tel:, javascript:...
        return None
    return p._replace(fragment='').geturl()  # remove o #ancora

def mesmo_dominio(a, b):
    """True se as duas URLs pertencem ao mesmo host."""
    return urlparse(a).netloc.lower() == urlparse(b).netloc.lower()

base = 'https://exemplo.org/docs/intro'
print(normalizar_url(base, '../api/v1'))      # resolve relativo
print(normalizar_url(base, '#secao'))          # vira a própria página, sem âncora
print(mesmo_dominio(base, 'https://exemplo.org/outro'))

https://exemplo.org/api/v1
https://exemplo.org/docs/intro
True


### Isolar o conteúdo central

Uma página tem muito além do texto útil: menu, cabeçalho, rodapé, barras laterais — repetidos em todas
as páginas. Se coletássemos isso, o corpus ficaria poluído de ruído. A função abaixo remove a **moldura**
e seleciona o **miolo** (`<main>`, `<article>`...).

> **Atenção à armadilha.** Padrões genéricos como `'menu'` são perigosos: podem casar com blocos de
> conteúdo legítimos (vimos isso com `tekup-service-menu`, que era navegação entre subpáginas). Por isso
> a lista usa termos **específicos**.

In [19]:
# Padrões ESPECÍFICOS de 'moldura' (evitam remover conteúdo por engano)
_LIXO_PADROES = (
    'navbar', 'main-menu', 'nav-menu', 'primary-menu', 'menu-principal',
    'dropdown', 'submenu', 'megamenu', 'offcanvas',
    'site-header', 'page-header', 'masthead', 'topbar',
    'site-footer', 'page-footer', 'rodape',
    'breadcrumb', 'cookie', 'consent', 'social-', 'share-',
    'newsletter', 'skip-link', 'modal', 'popup', 'pagination', 'sr-only',
)
_SELETORES_PRINCIPAIS = (
    'main', '[role="main"]', 'article',
    '#content', '#main', '#primary',
    '.entry-content', '.post-content', '.page-content', '.main-content',
)

def _parece_lixo(tag):
    if not hasattr(tag, 'get'):
        return False
    ident = ' '.join(filter(None, [
        tag.get('id', '') or '',
        ' '.join(tag.get('class', []) or []),
        tag.get('role', '') or '',
    ])).lower()
    return bool(ident) and any(p in ident for p in _LIXO_PADROES)

def isolar_conteudo_central(soup):
    """Devolve só o miolo da página, descartando navegação/layout."""
    soup = BeautifulSoup(str(soup), 'html.parser')  # trabalha numa cópia
    for tag in soup(['script', 'style', 'noscript', 'template', 'form', 'iframe']):
        tag.decompose()
    for tag in soup.find_all(['header', 'footer', 'nav', 'aside']):
        tag.decompose()
    for tag in list(soup.find_all(True)):
        if tag.parent is None:
            continue
        if _parece_lixo(tag):
            tag.decompose()
    for seletor in _SELETORES_PRINCIPAIS:
        alvo = soup.select_one(seletor)
        if alvo and alvo.get_text(strip=True):
            return alvo
    return soup.body or soup

print('Função de isolamento pronta.')

Função de isolamento pronta.


### Extrair título, texto, links e figuras

Com o miolo isolado, extraímos o que vai para o corpus. Note que **figuras** (imagens e diagramas) são
coletadas de dentro do conteúdo central — incluindo casos de *lazy-load* e SVG inline.

In [20]:
def extrair_titulo(soup):
    if soup.title and soup.title.string:
        return soup.title.string.strip()
    h1 = soup.find('h1')
    return h1.get_text(strip=True) if h1 else '(sem título)'

def extrair_texto(central):
    return ' '.join(central.get_text(separator=' ', strip=True).split())

def extrair_links(central, url_base):
    links = set()
    for a in central.find_all('a', href=True):
        u = normalizar_url(url_base, a['href'])
        if u:
            links.add(u)
    return sorted(links)

def _melhor_src(img):
    for attr in ('src', 'data-src', 'data-original', 'data-lazy-src'):
        v = img.get(attr)
        if v and not v.startswith('data:'):
            return v
    ss = img.get('srcset') or img.get('data-srcset')
    if ss:
        cands = [p.strip().split(' ')[0] for p in ss.split(',') if p.strip()]
        if cands:
            return cands[-1]
    return None

def extrair_figuras(central, url_base):
    figuras, vistos = [], set()
    def legenda_de(tag):
        fig = tag.find_parent('figure')
        cap = fig.find('figcaption') if fig else None
        return cap.get_text(' ', strip=True) if cap else ''
    for img in central.find_all('img'):
        src = _melhor_src(img)
        u = normalizar_url(url_base, src) if src else None
        if u and u not in vistos:
            vistos.add(u)
            figuras.append({'url': u, 'alt': (img.get('alt') or '').strip(),
                            'legenda': legenda_de(img), 'tipo': 'imagem'})
    for svg in central.find_all('svg'):
        rot = ''
        t = svg.find('title')
        if t and t.get_text(strip=True):
            rot = t.get_text(strip=True)
        elif svg.get('aria-label'):
            rot = svg.get('aria-label').strip()
        figuras.append({'url': None, 'alt': rot, 'legenda': legenda_de(svg), 'tipo': 'svg'})
    return figuras

def baixar_pagina(url, timeout=10):
    """Baixa e analisa uma página HTML. Retorna dict ou None (em erro / não-HTML)."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
    except requests.exceptions.SSLError as e:
        print(f'  [SSL] {url}: não foi possível validar o certificado do servidor.')
        print(f'       {e}')
        return None
    except requests.RequestException as e:
        print(f'  [ERRO] {url}: {e}')
        return None
    if 'text/html' not in resp.headers.get('Content-Type', ''):
        print(f'  [PULADO] não é HTML: {url}')
        return None
    soup = BeautifulSoup(resp.text, 'html.parser')
    central = isolar_conteudo_central(soup)
    return {'titulo': extrair_titulo(soup), 'texto': extrair_texto(central),
            'links': extrair_links(central, url), 'figuras': extrair_figuras(central, url)}

print('Funções de extração prontas.')

Funções de extração prontas.


### O crawler (busca em largura)

Tudo se junta aqui. Uma **fila** processa as páginas por nível (BFS). As salvaguardas — profundidade,
teto global, teto por domínio — são o que impede o crawl de rodar para sempre.

> **EXPERIMENTE (Exercícios 1 e 4).** Os parâmetros `profundidade_max`, `max_paginas` e
> `max_por_dominio` são os que você vai ajustar na prática. Observe como cada um limita o crawl.

In [21]:
def coletar(paginas_semente, profundidade_max=2, mesmo_dominio_only=True,
            max_paginas=150, max_por_dominio=40, delay=0.5, respeitar_robots=True):
    """Percorre as páginas (BFS) coletando conteúdo central, links e figuras."""
    visitadas, resultados, por_dominio = set(), [], Counter()
    fila = deque()
    for url in paginas_semente:
        u = normalizar_url(url, url)
        if u:
            fila.append((u, 0, None))

    while fila:
        if len(resultados) >= max_paginas:
            print(f'\n[LIMITE] teto global de {max_paginas} páginas atingido.')
            break
        url, prof, pai = fila.popleft()
        if url in visitadas:
            continue
        visitadas.add(url)
        dom = urlparse(url).netloc.lower()
        if max_por_dominio is not None and por_dominio[dom] >= max_por_dominio:
            continue
        if not pode_acessar(url, respeitar_robots):
            print(f'[BLOQUEADO robots.txt] {url}')
            continue
        print(f'[{len(resultados)+1}/{max_paginas}] nível {prof} | fila: {len(fila)} | {url}')
        dados = baixar_pagina(url)
        time.sleep(delay)
        if dados is None:
            continue
        por_dominio[dom] += 1
        resultados.append({
            'url': url, 'titulo': dados['titulo'], 'texto': dados['texto'],
            'relacao': 'raiz' if pai is None else 'filho', 'pai': pai,
            'profundidade': prof, 'links': dados['links'], 'figuras': dados['figuras'],
        })
        if profundidade_max is None or prof < profundidade_max:
            for link in dados['links']:
                if mesmo_dominio_only and not mesmo_dominio(url, link):
                    continue
                if link not in visitadas:
                    fila.append((link, prof + 1, url))
    return resultados

print('Crawler pronto.')

Crawler pronto.


---
## Bloco 4 — Qualidade: as sementes e os PDFs disfarçados

As sementes vêm de uma **planilha curada** (Google Sheets) com a coluna `source_url`. Dois cuidados:
ler a planilha de forma robusta e **descartar PDFs** — inclusive os disfarçados, que não terminam em
`.pdf` (ex.: `/content` da OMS, `/PDF/?uri=` do EUR-Lex).

A detecção tem duas camadas: uma **pista pela URL** (grátis) e, nos casos incertos, uma **verificação
real** via requisição HEAD (confere o `Content-Type`).

In [22]:
def _url_para_csv_export(url, gid=None):
    """Converte o link de uma planilha Google em URL de exportação CSV."""
    m = re.search(r'/spreadsheets/d/([a-zA-Z0-9-_]+)', url)
    if not m:
        raise ValueError('Link de Google Sheets inválido: ' + url)
    sid = m.group(1)
    if gid is None:
        g = re.search(r'[#&?]gid=([0-9]+)', url)
        gid = g.group(1) if g else None
    out = f'https://docs.google.com/spreadsheets/d/{sid}/export?format=csv'
    return out + (f'&gid={gid}' if gid is not None else '')

def _pista_de_pdf_na_url(url):
    """Heurística leve: True (é PDF), False (não), None (incerto -> checar de fato)."""
    if not url:
        return False
    u = url.lower()
    caminho = u.split('?')[0].split('#')[0]
    if caminho.endswith('.pdf') or '/pdf/' in caminho or '.pdf' in u.split('#')[0]:
        return True
    if caminho.endswith('/download') or 'attachment' in u or caminho.endswith('/content'):
        return None
    return False

def _eh_pdf_confirmado(url, timeout=10):
    try:
        r = requests.head(url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        return 'application/pdf' in r.headers.get('Content-Type', '').lower()
    except requests.RequestException:
        return False

def _eh_pdf(url, verificar_tipo_real=True):
    pista = _pista_de_pdf_na_url(url)
    if pista is True:
        return True
    if pista is False:
        return False
    return _eh_pdf_confirmado(url) if verificar_tipo_real else False

def ler_sementes_da_planilha(url_planilha, coluna='source_url', gid=None, verificar_tipo_real=True):
    """Lê a planilha pública (CSV), extrai `coluna` e devolve os links NÃO-PDF, sem duplicatas."""
    csv_url = _url_para_csv_export(url_planilha, gid)
    resp = requests.get(csv_url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    if 'text/csv' not in resp.headers.get('Content-Type', '') and '<html' in resp.text[:200].lower():
        raise PermissionError('A planilha não retornou CSV. Compartilhe como '
                              '"Qualquer pessoa com o link pode ver".')
    leitor = csv.DictReader(io.StringIO(resp.text))
    if coluna not in (leitor.fieldnames or []):
        raise KeyError(f'Coluna {coluna!r} não existe. Colunas: {leitor.fieldnames}')
    sementes, vistos, n_pdf = [], set(), 0
    for linha in leitor:
        u = (linha.get(coluna) or '').strip()
        if not u:
            continue
        if _eh_pdf(u, verificar_tipo_real):
            n_pdf += 1
            continue
        if u not in vistos:
            vistos.add(u)
            sementes.append(u)
    print(f'Sementes não-PDF: {len(sementes)} | PDFs descartados: {n_pdf}')
    return sementes

print('Leitura da planilha pronta.')

Leitura da planilha pronta.


> **EXPERIMENTE (Exercício 5).** Troque `verificar_tipo_real=False` e compare quantos PDFs são
> detectados. Quais links disfarçados passam batido sem a verificação real?


---
## Bloco 5 — Executando a coleta e salvando o corpus

Configure a planilha e os parâmetros, rode a coleta e salve o resultado em JSON. **Comece com valores
conservadores** (poucas páginas) para validar antes de uma coleta maior.

In [28]:
# === Configuração ===
URL_PLANILHA = 'https://docs.google.com/spreadsheets/d/1tv05ZRkyHhXSgX0XjdbSxNl9ssHJoBR2/edit?usp=sharing'  # planilha do corpus reduzido (13 documentos)

GID_ABA = None
COLUNA_URL = 'source_url'

PROFUNDIDADE_MAX = 1     # sementes + 1 níveis
MAX_PAGINAS      = 60    # comece baixo! aumente quando estiver confiante
MAX_POR_DOMINIO  = 10
DELAY_SEGUNDOS   = 0.5
ARQUIVO_SAIDA    = 'corpus_coletado.json'

# Para testar sem a planilha, você pode definir sementes manualmente:
# PAGINAS_SEMENTE = ['https://www.iana.org/help/example-domains']
PAGINAS_SEMENTE = ler_sementes_da_planilha(URL_PLANILHA, COLUNA_URL, GID_ABA)
print(f'{len(PAGINAS_SEMENTE)} semente(s) prontas.')

Sementes não-PDF: 9 | PDFs descartados: 4
9 semente(s) prontas.


In [29]:
resultados = coletar(
    PAGINAS_SEMENTE,
    profundidade_max=PROFUNDIDADE_MAX,
    max_paginas=MAX_PAGINAS,
    max_por_dominio=MAX_POR_DOMINIO,
    delay=DELAY_SEGUNDOS,
)

print(f'\nColetadas {len(resultados)} página(s).')
figs = sum(len(r['figuras']) for r in resultados)
prof = max((r['profundidade'] for r in resultados), default=0)
print(f'Profundidade máx.: {prof} | Figuras: {figs}')

[1/60] nível 0 | fila: 8 | https://unesdoc.unesco.org/ark:/48223/pf0000386276
[2/60] nível 0 | fila: 7 | https://www.iso.org/standard/56641.html
[3/60] nível 0 | fila: 12 | https://www.iso.org/obp/ui/en/
[4/60] nível 0 | fila: 10 | https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/lei/l13709.htm
[5/60] nível 0 | fila: 33 | https://sbis.org.br/certificacoes/certificacao-de-ia/
[6/60] nível 0 | fila: 37 | https://iris.who.int/server/api/core/bitstreams/e9e62c65-6045-481e-bd04-20e206bc5039/content
  [PULADO] não é HTML: https://iris.who.int/server/api/core/bitstreams/e9e62c65-6045-481e-bd04-20e206bc5039/content
[6/60] nível 1 | fila: 36 | https://www.iso.org/committee/6794475.html
[7/60] nível 1 | fila: 35 | https://www.iso.org/contact-iso.html
[8/60] nível 1 | fila: 34 | https://www.iso.org/contents/data/standard/05/66/56641.detail.rss
  [PULADO] não é HTML: https://www.iso.org/contents/data/standard/05/66/56641.detail.rss
[8/60] nível 1 | fila: 33 | https://www.iso.org/ics/35.020

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
with open(ARQUIVO_SAIDA, 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print(f'Corpus salvo em {ARQUIVO_SAIDA} ({len(resultados)} registros).')

# No Colab, para baixar:
# from google.colab import files; files.download(ARQUIVO_SAIDA)

Corpus salvo em corpus_coletado.json (22 registros).


---
## Bloco 6 — Da coleta ao RAG

O corpus está pronto. Cada registro foi pensado para alimentar um RAG:

| Campo | Papel no RAG |
|-------|--------------|
| `url` | **proveniência** — permite ao RAG citar a fonte |
| `texto` | vira os **chunks** que serão indexados e recuperados |
| `titulo` | rótulo e contexto do trecho |
| `pai` / `profundidade` | rastreabilidade do caminho de coleta |

A célula abaixo mostra um **esboço conceitual** do *chunking* — o primeiro passo do pipeline de RAG.
Não construímos o RAG inteiro aqui, mas vemos como o texto coletado se transforma
na entrada da próxima etapa.

> **EXPERIMENTE (Exercício 6).** Ajuste o tamanho do *chunk* e observe o efeito no número de trechos.
> Pense: trechos muito grandes ou muito pequenos — como afetam a recuperação?

In [31]:
def chunk_simples(texto, tamanho=800, sobreposicao=100):
    """Esboço didático de chunking por janela deslizante (em caracteres).

    Um chunker de produção respeitaria fronteiras de frase/parágrafo e contaria tokens,
    mas a ideia central é esta: fatiar o texto em pedaços recuperáveis, com leve sobreposição
    para não perder contexto nas bordas.
    """
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fim = inicio + tamanho
        chunks.append(texto[inicio:fim])
        inicio = fim - sobreposicao
    return chunks

# Demonstra sobre o que foi coletado
if resultados:
    exemplo = resultados[0]
    pedacos = chunk_simples(exemplo['texto'])
    print(f'Página: {exemplo["titulo"][:60]}')
    print(f'Texto: {len(exemplo["texto"])} caracteres -> {len(pedacos)} chunks')
    print(f'\nPrimeiro chunk (com proveniência para citação):')
    print(f'  fonte: {exemplo["url"]}')
    for chunk in pedacos:
      print(f'  trecho: {chunk[:200]}...')
else:
    print('Rode a coleta primeiro (Bloco 5).')

Página: (sem título)
Texto: 0 caracteres -> 0 chunks

Primeiro chunk (com proveniência para citação):
  fonte: https://unesdoc.unesco.org/ark:/48223/pf0000386276


---
## Síntese

Percorremos as cinco decisões de engenharia que transformam um script ingênuo num scraper confiável:

1. **Abordagem** — scraping estático, adequado às nossas fontes.
2. **Ética** — robots.txt, rate limiting, User-Agent honesto.
3. **Escopo** — BFS com profundidade e salvaguardas.
4. **Qualidade** — conteúdo central isolado, PDFs (mesmo disfarçados) filtrados.
5. **Saída para RAG** — estrutura com proveniência rastreável.

A lição que atravessa tudo: **a qualidade do RAG é limitada pela qualidade da coleta**. Cada decisão
aqui tem consequência lá na frente, quando o sistema recuperar trechos e gerar respostas.
